<a href="https://colab.research.google.com/github/general-molecular-simulations/so3lr/blob/main/examples/so3lr_colab_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Install SO3LR
#@markdown ## Change runtime type to GPU (if available), then click run. Installation takes ~2 minutes.
import os
import sys

_is_so3lr_setup_file = '/content/.SO3LR_SETUP'

if not os.path.exists(_is_so3lr_setup_file):
  # Install Miniconda with Python 3.12
  os.system('wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh')
  os.system('bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local')
  os.system('conda install -y -q --prefix /usr/local python=3.12')

  # Install JAX (try GPU first, fall back to CPU)
  os.system('uv pip install --upgrade pip')
  if os.system('uv pip install "jax[cuda12]==0.5.3"') != 0:
    os.system('uv pip install jax==0.5.3')

  # Install dependencies
  os.system('uv pip install ase py3Dmol')

  # Update paths
  sys.path.append('/usr/local/lib/python3.12/site-packages/')
  os.environ['CONDA_PREFIX'] = '/usr/local/'
  os.environ['CONDA_DEFAULT_ENV'] = 'base'

  # Install SO3LR
  os.system('git clone https://github.com/general-molecular-simulations/so3lr.git /content/so3lr')
  os.system('cd /content/so3lr && uv pip install .')

  # Cleanup
  os.system(f"touch {_is_so3lr_setup_file}")
  os.unlink('Miniconda3-latest-Linux-x86_64.sh')
else:
  # Just verify imports
  import jax
  import so3lr
  print(f"SO3LR ready. JAX backend: {jax.default_backend()}")


## Running simulations

SO3LR ships a small command-line interface; this notebook wraps it in interactive forms.

1. Run the **Install SO3LR** cell above once per session (set the runtime to GPU first).
2. Use **Run an SO3LR simulation** to optimize a geometry (`opt`), run molecular
   dynamics (`nvt`, `npt`, `nve`), or evaluate energies/forces on a dataset (`eval`).
3. Use **Visualize a trajectory** to animate the resulting `.xyz`.

The bundled example `/content/so3lr/examples/data/ala15_folded.xyz` is a folded Ala15
peptide. To run your own system, upload an ASE-readable structure file via the **Files**
panel (the folder icon on the left) and set its path as the input.


In [ ]:
#@title Run an SO3LR simulation
#@markdown Pick a simulation type, point to an input structure, set the parameters, and run.

simulation_type = "nvt" #@param ["opt", "nvt", "npt", "nve", "eval"]
input_file = "/content/so3lr/examples/data/ala15_folded.xyz" #@param {type:"string"}
output_file = "" #@param {type:"string"}
model = "so3lr-m" #@param ["so3lr-s", "so3lr-m", "so3lr-l"]

#@markdown **Common options**
lr_cutoff = 12.0 #@param {type:"number"}
total_charge = 0 #@param {type:"integer"}

#@markdown **Molecular dynamics only** (`nvt` / `npt` / `nve`) — temperature in K, timestep in fs
temperature = 300.0 #@param {type:"number"}
timestep_fs = 0.5 #@param {type:"number"}
md_cycles = 50 #@param {type:"integer"}
md_steps = 20 #@param {type:"integer"}
seed = 0 #@param {type:"integer"}
pressure_atm = 1.0 #@param {type:"number"}

#@markdown **Optimization only** (`opt`) — force convergence in eV/Å
force_conv = 0.05 #@param {type:"number"}

#@markdown **Evaluation only** (`eval`) — comma-separated targets, e.g. `forces,dipole_vec`
eval_targets = "forces" #@param {type:"string"}

import os
import shlex
import subprocess

if not output_file:
    stem = os.path.splitext(os.path.basename(input_file))[0]
    output_file = f"{stem}_{simulation_type}.xyz"

if simulation_type == "eval":
    cmd = ["so3lr", "eval", "--datafile", input_file, "--save-to", output_file,
           "--model", model, "--lr-cutoff", str(lr_cutoff)]
    if eval_targets.strip():
        cmd += ["--targets", eval_targets.strip()]
else:
    cmd = ["so3lr", simulation_type, "--input", input_file, "--output", output_file,
           "--model", model, "--lr-cutoff", str(lr_cutoff),
           "--total-charge", str(total_charge)]
    if simulation_type == "opt":
        cmd += ["--force-conv", str(force_conv)]
    else:  # nvt / npt / nve
        cmd += ["--temperature", str(temperature), "--dt", str(timestep_fs),
                "--md-cycles", str(md_cycles), "--md-steps", str(md_steps),
                "--seed", str(seed)]
        if simulation_type == "npt":
            cmd += ["--pressure", str(pressure_atm)]

print("Running:\n  " + " ".join(shlex.quote(c) for c in cmd) + "\n" + "-" * 70)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
code = proc.wait()
print("-" * 70)
if code == 0:
    print(f"Done. Output written to: {output_file}")
else:
    print(f"SO3LR exited with code {code}. Check the log above.")


In [ ]:
#@title Visualize a trajectory
#@markdown Path to an `.xyz` trajectory (e.g. the output of the cell above). Multi-frame files are animated.

trajectory_file = "ala15_folded_nvt.xyz" #@param {type:"string"}

import os
from io import StringIO

import ase.io
import py3Dmol

assert os.path.exists(trajectory_file), f"File not found: {trajectory_file}"

frames = list(ase.io.read(trajectory_file, index=":"))
print(f"Loaded {len(frames)} frame(s) from {trajectory_file}")

buf = StringIO()
for atoms in frames:
    ase.io.write(buf, atoms, format="xyz")

view = py3Dmol.view(width=800, height=500)
view.addModel(buf.getvalue(), "xyz", {"multimodel": True})
view.setStyle({"sphere": {"radius": 0.4}, "stick": {"radius": 0.2}})
view.setBackgroundColor("white")
view.zoomTo()
if len(frames) > 1:
    view.animate({"loop": "forward", "interval": 100})
view.show()
